# NB03 — Sintonía diferenciable del FOPID a través del sustituto identificado

**Proyecto:** fPINN + sintonía diferenciable de FOPID en el TCLab — *Control Engineering Practice*.
**Secciones del manuscrito:** 3.3–3.4 (lazo cerrado diferenciable y costo con robustez), 4.3 (ablaciones y
transferencia) y la tabla de diseño `tab:design`. Los controladores que salen de aquí son los que NB04 prueba en el kit.

**Qué hace.**
1. Carga el sustituto identificado por NB02 (ϑ̂ del modelo 2×2 en JSON; la red se descarta) y los modelos SISO
   (SOPDT, FFOPDT) de la misma partición.
2. Simula el lazo cerrado T1–Q1 con Q2 como perturbación, con el FOPID en forma GL discreta (integral fraccionaria
   del error con anti-windup por back-calculation, derivada fraccionaria filtrada sobre la medición, saturación 0–100 %),
   **todo en PyTorch**, así que el costo es diferenciable respecto a p = (Kp, Ki, Kd, λ, μ, Kt).
3. Costo = Σ_j [IAE referencia + IAE perturbación Q2 + TV(u)] (normalizados con B4) + ρ·([Ms − Ms_max]₊² + [PM_min − PM]₊²),
   con Ms y PM calculados en frecuencia sobre la linealización exacta del modelo fraccionario en cada punto de operación.
4. Sintoniza P (Adam → L-BFGS, arranque en el PID SIMC con λ = μ = 1) y los comparadores:

| ID | Controlador | Modelo de diseño | Sintonía |
|---|---|---|---|
| **P** | FOPID | sustituto de diseño (`DESIGN_MODEL`) | gradiente, (Kp, Ki, Kd, λ, μ, Kt) |
| **B1** | FOPID | FFOPDT | especificaciones de Monje (ωc, PM, fase plana, Ms) |
| **B2** | FOPID | el mismo sustituto | PSO, mismo costo |
| **B3** | PID | PINN entera | gradiente, λ = μ = 1 |
| **B4** | PID | SOPDT | SIMC, τc tal que Ms = Ms_max |
| **A1** | PID / FOPID | sustituto | λ = μ = 1 fijos · λ = μ = regla · aprendibles (= P) |
| **A2** | FOPID | sustituto | sin J_rob (ρ = 0) |

5. Evalúa todos los controladores en el sustituto de diseño **y en plantas de verificación** (otros modelos
   identificados: el físico entero OE-LS —verificación de Ms y PM pedida por Luis—, la PINN entera y las fPINN de las
   particiones AB-C y BC-A), calcula ΔJ (transferencia) y aplica la
   **regla de decisión**: si P no supera claramente a B4 en perturbación o robustez → evaluar 2×2 MIMO en simulación.

**Modo `SOURCE`.** `'kit'` (principal): sustitutos de `NB02_fPINN_kit.ipynb`. `'sim'`: sustitutos de
`NB02_fPINN_simulacion.ipynb`; la planta de verificación es entonces la **planta virtual fraccionaria** (ϑ real conocido),
que es lo que pide la sección 4.3 del manuscrito.

**Cómo usarlo.** `MODE = 'quick'` (prueba, ~2–4 min: 2 puntos de operación, horizontes cortos, pocas iteraciones).
`MODE = 'full'`: 3 puntos de operación, 900 + 900 s por punto, 3 arranques de P, PSO completo. Tiempo estimado en una
CPU de 4 núcleos: **~1–2 h** (lo que más pesa es B2 y los arranques de P). Cada controlador se guarda en
`NB03_out/<SOURCE>/controladores.json` en cuanto termina y **se retoma** si se interrumpe. Al final crea
`NB03_<SOURCE>_resultados_<fecha>.zip` con todo `NB03_out/<SOURCE>/`. Requiere PyTorch (`%pip install torch`).

In [ ]:
import os
# Windows/Anaconda: torch y numpy traen cada uno su OpenMP (libiomp5md.dll); sin esto el kernel muere sin mensaje.
# Debe ir ANTES de importar numpy y torch.
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
import json, time, math, glob, zipfile, datetime, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import fsolve, brentq
import torch
torch.set_default_dtype(torch.float64)

MODE = 'full'               # 'quick' (prueba) | 'full' (corrida completa)
SOURCE = 'kit'               # 'kit' (sustitutos de NB02_fPINN_kit) | 'sim' (sustitutos de NB02_fPINN_simulacion)

# ------------------------------------------------------------------------------------------------------------------
# RUTAS: lo único que hay que tocar cuando cambien los resultados de NB02.
# NB02_DIR = None busca, en este orden: NB02_out/kit/ (o NB02_out/ en 'sim') en esta carpeta o en '..', y si no está,
# el NB02_kit_resultados_*.zip más reciente (prefiere el que NO termina en _quick) y lo descomprime.
# También se puede poner una carpeta concreta (la que contiene surrogates/) o un .zip concreto, p. ej.
#   NB02_DIR = r'C:\TCLab\NB02_kit_resultados_20261109.zip'
NB02_DIR = os.path.join('..','data','NB02_kit_20261109_ABC','NB02_out','kit')
# Modelo de diseño (P, B2, A1, A2): archivo de sustitutos (partición_ambiente_semilla, sin .json) + clave del modelo.
#   Decisión de Luis (9 nov): fPINN reentrenada con A+B+C. DESIGN_TAG = 'auto' toma el JSON de surrogates/ cuyo
#   'train' es A, B y C con ambiente 'const' (semilla más baja); si no existe todavía, usa 'AB-C_const_s0' y avisa.
#   Para fijarlo a mano: p. ej. DESIGN_TAG = 'ABC_const_s0' (el nombre exacto del archivo en surrogates/).
#   DESIGN_MODEL: 'fPINN' (propuesto) | 'IntegerPINN' | 'PhysInt_OE' | 'PhysFrac_OE'.
DESIGN_TAG, DESIGN_MODEL = 'ABC_const_s0', 'PhysInt_OE'
B3_MODEL = 'IntegerPINN'     # modelo de diseño de B3 (mismo JSON)
# Plantas de verificación (tag, modelo): 'DESIGN' = el mismo JSON del diseño. Las que no existan se saltan.
#   El físico entero OE-LS va primero: Luis pidió verificar Ms y PM del FOPID también sobre ese modelo.
VERIF = [('DESIGN', 'PhysInt_OE'), ('DESIGN', 'IntegerPINN'), ('AB-C_const_s0', 'fPINN'), ('BC-A_const_s0', 'fPINN')]
# Solo SOURCE='sim': sustituto de diseño y caso real de la planta virtual
SIM_TAG, SIM_CASE = 'a0.9_b0.7_s0', (0.9, 0.7)
# ------------------------------------------------------------------------------------------------------------------

OUT_DIR = os.path.join('NB03_out', SOURCE)
os.makedirs(OUT_DIR, exist_ok=True)
N_THREADS = None             # p. ej. 4; None = valor por defecto de PyTorch
if N_THREADS: torch.set_num_threads(N_THREADS)

TS = 1.0                     # periodo de muestreo del kit [s]
SC = dict(ops=[35.0, 50.0, 65.0],   # puntos de operación T̄_j [°C] (en 'sim' se restan 5 °C: ambiente 23 °C)
          dr=5.0,            # escalón de referencia [°C]
          dq2=40.0,          # escalón de perturbación en Q2 [%]
          t_r=900, t_d=900,  # duración de cada tramo [s]: referencia en t = 10 s, perturbación en t = t_r
          noise=0.03,        # ruido de medición del controlador [°C] (NB01: ≈ 0,03 °C; secuencia fija)
          noise_seed=123)
CTRL = dict(Tf=10.0)         # filtro de la derivada [s] (fijo, igual para todos). Decisión de Luis (9 nov): 10 s.
                             # Con 5 s (primera corrida) Q1 tenía ±5–10 % de ruido (Kd ≈ 500). Cambiarlo rehace todo.
ROB = dict(Ms_max=1.6, PM_min=45.0, kappa=200.0, n_w=400, w_min=1e-4)
# Dónde se fija Ms = Ms_max de B4 (τc del SIMC) y la cota de Ms de B1:
#   'both' (por defecto): Ms <= Ms_max a la vez sobre su propio modelo (SOPDT para B4, FFOPDT para B1) y sobre la
#            linealización del sustituto de diseño (peor punto de operación), la referencia común de P, B2, B3 y A1–A2.
#   'design': solo sobre el sustituto (en la prueba dio un SIMC con τc = 5 s, Ms = 3,1 sobre su propio SOPDT).
#   'own'   : solo sobre su propio modelo, como lo haría un usuario sin el sustituto.
MS_REF = 'both'
MS_REF_B1 = 'own'            # B1 aísla el efecto del modelo: se diseña solo con el FFOPDT (como Monje et al.)
MS_REF_B3 = 'both'           # B3: cota de Ms/PM sobre la PINN entera y TAMBIÉN sobre el sustituto de diseño (como B4).
                             # 'own' = solo PINN entera (corrida del 9 nov: Ms 1,95 / PM 31° en la fPINN).
COST = dict(w_r=1.0, w_d=1.0, w_u=0.2, rho=100.0)    # pesos de J_perf (normalizados con B4) y de J_rob
OPT = dict(n_adam=300, lr=0.05, n_lbfgs=40, n_starts=3, n_starts_other=1, start_sd=0.3, seed=0)  # arranques: P / resto
PSO = dict(n_part=40, n_iter=100, w=0.7, c1=1.5, c2=1.5, seed=0)
B1_SPEC = dict(PM=60.0, flat=True, T_hf_dB=-20.0, S_lf_dB=-20.0)   # especificaciones tipo Monje (2008); ωc se busca
ORDER_RULE = 'alpha_hat'     # A1 "regla": λ = μ = α̂ del sustituto
DECISION_GAIN = 10.0         # % de mejora de IAE_d que cuenta como "claramente mejor"
DECISION_MS = 0.1            # o Ms menor en al menos esto, a IAE_d igual o mejor

if MODE == 'quick':
    SC.update(ops=[40.0, 55.0], t_r=500, t_d=500)
    OPT.update(n_adam=15, n_lbfgs=3, n_starts=1, n_starts_other=1)
    PSO.update(n_part=6, n_iter=3)
    ROB.update(n_w=250)
if SOURCE == 'sim':
    SC['ops'] = [o - 5.0 for o in SC['ops']]
    VERIF = []               # se reemplaza por la planta virtual
print('MODE =', MODE, '| SOURCE =', SOURCE, '| diseño:', DESIGN_TAG if SOURCE == 'kit' else SIM_TAG, DESIGN_MODEL,
      '| puntos de operación', SC['ops'])

## 1. Carga de los sustitutos de NB02

Cada JSON de `surrogates/` trae `phys` (constantes fijas), `h` y `models` con ϑ̂ de cada modelo: fPINN, PINN entera,
físico entero/fraccionario (OE-LS), FOPDT, SOPDT y FFOPDT. En `'kit'` el modelo tiene **Us** (intercambio entre
calentadores) separado de U; en `'sim'` no, y se usa Us = U.

In [ ]:
def find_nb02(nb02_dir=None):
    sub = os.path.join('NB02_out', 'kit') if SOURCE == 'kit' else 'NB02_out'
    has = lambda d: os.path.isdir(os.path.join(d, 'surrogates'))
    if nb02_dir and os.path.isdir(nb02_dir):
        for d in [nb02_dir, os.path.join(nb02_dir, sub)]:
            if has(d): return d, d
        raise FileNotFoundError(f'{nb02_dir} no contiene surrogates/')
    if not nb02_dir:
        for d in [sub, os.path.join('..', sub)]:
            if has(d): return d, d
    if nb02_dir and nb02_dir.endswith('.zip'):
        zips = [nb02_dir]
    else:
        pat = 'NB02_kit_resultados_*.zip' if SOURCE == 'kit' else 'NB02_*resultados*.zip'
        zips = sorted(glob.glob(pat) + glob.glob(os.path.join('..', pat)), key=os.path.getmtime)
        full = [z for z in zips if not z.endswith('_quick.zip')]
        zips = full or zips
        abc = [z for z in zips if '_ABC' in os.path.basename(z)]      # preferir el zip del entrenamiento final A+B+C
        zips = abc or zips
    if zips:
        z = zips[-1]; dest = os.path.join('NB03_out', 'NB02_desde_zip')
        with zipfile.ZipFile(z) as f: f.extractall(dest)
        for root, dirs, _ in os.walk(dest):
            if 'surrogates' in dirs: return root, z
    raise FileNotFoundError('No se encontró NB02_out/ ni un zip de NB02 con surrogates/. Poner NB02_DIR.')

SUR_DIR, SUR_SRC = find_nb02(NB02_DIR)
def load_sur(tag):
    return json.load(open(os.path.join(SUR_DIR, 'surrogates', f'{tag}.json')))

def auto_tag():
    cands = []
    for fn in sorted(glob.glob(os.path.join(SUR_DIR, 'surrogates', '*.json'))):
        js = json.load(open(fn)); tr = js.get('train') or []
        if sorted(tr) == ['A', 'B', 'C'] and js.get('amb', 'const') == 'const' and DESIGN_MODEL in js.get('models', {}):
            cands.append((js.get('seed', 0), os.path.basename(fn)[:-5]))
    if cands: return sorted(cands)[0][1]
    print('AVISO: no hay sustituto entrenado con A+B+C en', SUR_DIR, '-> se usa AB-C_const_s0 (cambiar cuando llegue)')
    return 'AB-C_const_s0'

if SOURCE == 'kit' and DESIGN_TAG == 'auto': DESIGN_TAG = auto_tag()
D_JSON = load_sur(DESIGN_TAG if SOURCE == 'kit' else SIM_TAG)
print('Sustituto de diseño:', DESIGN_TAG if SOURCE == 'kit' else SIM_TAG, '/', DESIGN_MODEL, '| entrenado con', D_JSON.get('train'))
print('Sustitutos de NB02 tomados de:', SUR_SRC)
print('Archivos:', sorted(os.listdir(os.path.join(SUR_DIR, 'surrogates'))))
for k, v in D_JSON['models'].items():
    print(f'  {k:12s}', {a: round(b, 4) for a, b in v.items()})

## 2. Modelo de planta (el de NB02) en torch, equilibrio y linealización

En variables de desviación z = x − x̄ respecto al **equilibrio** del punto de operación (x̄ resuelve F(x̄, Q̄) = 0 con
T_S1 = T̄), la derivada de Caputo con historia constante en x̄ es exactamente el GL aplicado a z, así que el lazo arranca
en equilibrio sin transitorio artificial:
z_H,k = h^α/(m cp t_c^(α−1))·F(x̄ + z_H,k−1, Q_k−1) − Σ_{i≥1} w_i^(α) z_H,k−i ;  z_S,k = (h/τs)^β (z_H,k−1 − z_S,k−1) − Σ w_i^(β) z_S,k−i
(memoria completa, igual que `sim_frac` de NB02). La función de transferencia G11(jω) se obtiene de la linealización
**exacta** del modelo 2×2 fraccionario en (x̄, Q̄) con Q2 constante (incluye el acoplamiento a través del calentador 2),
más el retardo de medio periodo del retenedor: G11 = e1ᵀ(c (jω)^α I − Jx)⁻¹ b · 1/((jωτs)^β + 1) · e^(−jωh/2).

In [ ]:
class Plant:
    def __init__(self, th, phys, Ta, name):
        self.th = dict(th); self.th.setdefault('Us', self.th['U'])
        self.phys = dict(phys); self.phys.setdefault('tc', 1.0)
        self.Ta, self.TaK, self.name = float(Ta), float(Ta) + 273.15, name
        self.mcp = self.phys['m'] * self.phys['cp']

    def F(self, x1, x2, q1, q2):
        # balance de energía (numpy o torch); x = T − T∞ [K]
        th, p = self.th, self.phys
        es, A, As, Ta = p['eps'] * p['sigma'], p['A'], p['As'], self.TaK
        T1, T2 = Ta + x1, Ta + x2
        ex = th['Us'] * As * (T2 - T1) + es * As * (T2**4 - T1**4)
        return (th['U'] * A * (Ta - T1) + es * A * (Ta**4 - T1**4) + ex + th['alpha1'] * q1,
                th['U'] * A * (Ta - T2) + es * A * (Ta**4 - T2**4) - ex + th['alpha2'] * q2)

    def equilibrium(self, Tbar, q2=0.0):
        # incógnitas (Q1, x2) con x1 = T̄ − T∞; en equilibrio x_S = x_H
        x1 = Tbar - self.Ta
        sol, info, ok, msg = fsolve(lambda v: self.F(x1, v[1], v[0], q2), [x1 / 0.75, 0.3 * x1], full_output=True)
        if ok != 1: print('Aviso equilibrio', self.name, Tbar, msg)
        q1, x2 = sol
        return dict(Tbar=Tbar, u=float(q1), q2=float(q2), x=np.array([x1, x2, x1, x2]))

    def G11(self, w, eq, h=TS):
        x1, x2 = eq['x'][:2]; d = 1e-4
        f = lambda a, b: np.array(self.F(a, b, eq['u'], eq['q2']))
        Jx = np.c_[(f(x1 + d, x2) - f(x1 - d, x2)) / (2 * d), (f(x1, x2 + d) - f(x1, x2 - d)) / (2 * d)]
        al, be, ts_ = self.th['alpha'], self.th['beta'], self.th['tau_s']
        c = self.mcp * self.phys['tc'] ** (al - 1); sa = (1j * w) ** al
        M11, M12, M21, M22 = c * sa - Jx[0, 0], -Jx[0, 1], -Jx[1, 0], c * sa - Jx[1, 1]
        heat = M22 * self.th['alpha1'] / (M11 * M22 - M12 * M21)
        return heat / ((1j * w * ts_) ** be + 1) * np.exp(-1j * w * h / 2)

def plant_from(js, model, name=None, Ta=None):
    Ta = Ta if Ta is not None else js.get('T_AMB_NB01') or js['phys'].get('Ta')
    return Plant(js['models'][model], js['phys'], Ta, name or model)

def sopdt_G(p, w, h=TS):
    return p['K'] * np.exp(-1j * w * (p['theta'] + h / 2)) / ((1j * w * p['tau1'] + 1) * (1j * w * p['tau2'] + 1))

def ffopdt_G(p, w, h=TS):
    al = p.get('alpha_f', p.get('alpha'))
    return p['K'] * np.exp(-1j * w * (p['theta'] + h / 2)) / (p['tau_a'] * (1j * w) ** al + 1)

# Planta de diseño, B3 y verificación
if SOURCE == 'kit':
    PL_D = plant_from(D_JSON, DESIGN_MODEL, f'{DESIGN_MODEL} {DESIGN_TAG}')
    PL_B3 = plant_from(D_JSON, B3_MODEL, f'{B3_MODEL} {DESIGN_TAG}')
    PL_V = []
    for t, m in VERIF:
        t = DESIGN_TAG if t == 'DESIGN' else t
        if (t, m) == (DESIGN_TAG, DESIGN_MODEL): continue
        if not os.path.exists(os.path.join(SUR_DIR, 'surrogates', f'{t}.json')):
            print('Verificación saltada (no existe):', t); continue
        js = load_sur(t)
        if m not in js['models']: print('Verificación saltada (sin modelo):', t, m); continue
        PL_V.append(plant_from(js, m, f'{m} {t}', Ta=PL_D.Ta))
else:
    PL_D = plant_from(D_JSON, DESIGN_MODEL, f'{DESIGN_MODEL} {SIM_TAG}')
    PL_B3 = plant_from(D_JSON, B3_MODEL, f'{B3_MODEL} {SIM_TAG}')
    TRUE = dict(U=5.0, alpha1=0.01, alpha2=0.0075, tau_s=20.0, alpha=SIM_CASE[0], beta=SIM_CASE[1])  # = NB02 sim
    PL_V = [Plant(TRUE, D_JSON['phys'], D_JSON['phys']['Ta'], f'planta virtual {SIM_CASE}')]
SOPDT, FFOPDT = D_JSON['models']['SOPDT'], D_JSON['models']['FFOPDT']
ALPHA_HAT = PL_D.th['alpha']
W = np.logspace(np.log10(ROB['w_min']), np.log10(np.pi / TS), ROB['n_w'])
W_DENSE = np.logspace(np.log10(ROB['w_min']), np.log10(np.pi / TS), 3000)

for pl in [PL_D, PL_B3] + PL_V:
    eqs = [pl.equilibrium(T) for T in SC['ops']]
    k0 = [abs(pl.G11(np.array([1e-7]), e)[0]) for e in eqs]
    print(f'{pl.name:28s} T∞={pl.Ta:.2f} | Q̄1 = ' + ', '.join(f'{e["u"]:.1f}' for e in eqs) +
          ' % | K(0) = ' + ', '.join(f'{k:.3f}' for k in k0) + ' °C/%')
print('SOPDT (B4):', {k: round(v, 3) for k, v in SOPDT.items()}, '| FFOPDT (B1):', {k: round(v, 3) for k, v in FFOPDT.items()})

## 3. Lazo cerrado diferenciable (FOPID GL + anti-windup) en lote

Por cada muestra k (h = 1 s), para un lote de B = (controladores × puntos de operación):
- e_k = r_k − y_k, con y_k = T_S1 + ruido fijo (σ = `SC['noise']`); ẽ_k = e_k + Kt (u_k−1 − v_k−1)  (back-calculation, ec. `eq:aw`)
- I_k = h^λ Σ_{i=0}^{k} w_i^(−λ) ẽ_k−i  (integral fraccionaria GL, memoria completa)
- D_k = filtro_Tf( h^−μ Σ w_i^(μ) (y_k−i − T̄) )  (derivada fraccionaria **sobre la medición**, filtro de 1.er orden, Euler implícito)
- v_k = Q̄1 + Kp e_k + Ki I_k − Kd D_k ;  u_k = sat_[0,100](v_k)   (Q̄1 = entrada de equilibrio: arranque sin salto)
- Referencia: T̄ → T̄ + `dr` en t = 10 s; perturbación: Q2 = 0 → `dq2` en t = `t_r`.

Las métricas usan el error **sin ruido**: IAE_r (tramo de referencia), IAE_d (tramo de perturbación) y TV(u) = Σ|Δu|.
Para NB04 se exporta exactamente esta ley de control.

In [ ]:
def gl_w_t(a, n):
    # pesos GL w_0..w_{n-1}; a escalar o tensor (B,) -> (n,) o (B, n)
    a = torch.as_tensor(a, dtype=torch.float64); k = torch.arange(1, n, dtype=torch.float64)
    if a.dim() == 0:
        return torch.cumprod(torch.cat([torch.ones(1), 1.0 - (a + 1.0) / k]), 0)
    return torch.cumprod(torch.cat([torch.ones(a.shape[0], 1), 1.0 - (a[:, None] + 1.0) / k[None, :]], 1), 1)

def scenario(N_r=None, N_d=None):
    N_r = N_r or SC['t_r']; N_d = N_d or SC['t_d']; N = N_r + N_d
    k = np.arange(N)
    rng = np.random.default_rng(SC['noise_seed'])
    return dict(N=N, N_r=N_r, r_step=(k >= 10).astype(float) * SC['dr'], q2_step=(k >= N_r).astype(float) * SC['dq2'],
                noise=rng.normal(0, SC['noise'], N))

SCN = scenario()

def closed_loop(plant, P, eqs, scn=SCN, u_open=None):
    # P: tensor (C, 6) en unidades naturales (Kp, Ki, Kd, lam, mu, Kt). eqs: lista de J equilibrios.
    # Devuelve dict con y (sin ruido), u, e (C, J, N). Si u_open (J, N) se da, simula en lazo abierto (chequeo).
    C, J, N = P.shape[0], len(eqs), scn['N']; B = C * J
    rep = lambda v: v.repeat_interleave(J)          # (C,) -> (B,) con el punto de operación más rápido
    Kp, Ki, Kd, lam, mu, Kt = (rep(P[:, i]) for i in range(6))
    Tbar = torch.tensor([e['Tbar'] for e in eqs]).repeat(C)
    ueq = torch.tensor([e['u'] for e in eqs]).repeat(C)
    q2eq = torch.tensor([e['q2'] for e in eqs]).repeat(C)
    xeq = torch.tensor(np.array([e['x'] for e in eqs])).repeat(C, 1)
    th, al, be = plant.th, plant.th['alpha'], plant.th['beta']
    ca = TS ** al / (plant.mcp * plant.phys['tc'] ** (al - 1)); cb = (TS / th['tau_s']) ** be
    wa, wb = gl_w_t(al, N + 1), gl_w_t(be, N + 1)
    wI, wD = gl_w_t(-lam, N + 1), gl_w_t(mu, N + 1)          # (B, N+1), diferenciables en λ, μ
    hI, hD = TS ** lam, TS ** (-mu)
    a_f = CTRL['Tf'] / (CTRL['Tf'] + TS)
    r = torch.tensor(scn['r_step']); q2s = torch.tensor(scn['q2_step']); nz = torch.tensor(scn['noise'])
    if u_open is not None: u_open = torch.as_tensor(u_open).repeat(C, 1)

    zH_hist = torch.zeros(B, 0, 2); zS_hist = torch.zeros(B, 0, 2)   # historia invertida (más reciente primero)
    E_hist = torch.zeros(B, 0); Y_hist = torch.zeros(B, 0)
    zH = torch.zeros(B, 2); zS = torch.zeros(B, 2)
    u_prev, v_prev, dflt = ueq.clone(), ueq.clone(), torch.zeros(B)
    ys, us, es = [], [], []
    for k in range(N):
        y = Tbar + zS[:, 0]; ym = y + nz[k]
        rk = Tbar + r[k]; e = rk - ym
        if u_open is None:
            et = e + Kt * (u_prev - v_prev)
            E_hist = torch.cat([et[:, None], E_hist], 1)
            I = hI * (wI[:, :k + 1] * E_hist).sum(1)
            Y_hist = torch.cat([(ym - Tbar)[:, None], Y_hist], 1)
            Dm = hD * (wD[:, :k + 1] * Y_hist).sum(1)
            dflt = a_f * dflt + (1 - a_f) * Dm
            v = ueq + Kp * e + Ki * I - Kd * dflt
            u = torch.clamp(v, 0.0, 100.0)
        else:
            v = u = u_open[:, k]
        ys.append(y); us.append(u); es.append(rk - y)
        # planta: estado k+1 con entradas del intervalo k
        x = xeq + torch.cat([zH, zS], 1)
        f1, f2 = plant.F(x[:, 0], x[:, 1], u, q2eq + q2s[k])
        zH_hist = torch.cat([zH[:, None, :], zH_hist], 1); zS_hist = torch.cat([zS[:, None, :], zS_hist], 1)
        mH = torch.einsum('n,bns->bs', wa[1:k + 2], zH_hist); mS = torch.einsum('n,bns->bs', wb[1:k + 2], zS_hist)
        zS = cb * (zH - zS) - mS
        zH = ca * torch.stack([f1, f2], 1) - mH
        u_prev, v_prev = u, v
    sh = lambda L: torch.stack(L, 1).reshape(C, J, N)
    return dict(y=sh(ys), u=sh(us), e=sh(es), ueq=ueq.reshape(C, J))

def perf_terms(out, scn=SCN):
    N_r = scn['N_r']; e, u = out['e'], out['u']
    du = torch.diff(torch.cat([out['ueq'][..., None], u], -1), dim=-1)
    return dict(IAE_r=TS * e[..., :N_r].abs().sum(-1), IAE_d=TS * e[..., N_r:].abs().sum(-1), TV=du.abs().sum(-1))

### Chequeo: el simulador en torch reproduce el GL de NB02

Lazo abierto desde el equilibrio a T̄ con Q1 = Q̄1 + escalones: (i) con Q1 = Q̄1 la temperatura no se mueve;
(ii) desde el ambiente (T̄ = T∞, Q̄ = 0) coincide con `sim_frac` de NB02 (copiado abajo, numpy).

In [ ]:
def gl_w(a, n):
    return np.cumprod(np.r_[1.0, 1.0 - (a + 1.0) / np.arange(1, n)])

def sim_frac_np(plant, Q1, Q2):
    # = sim_frac de NB02 (kit), ambiente constante, memoria completa
    th = plant.th; al, be = th['alpha'], th['beta']
    ca = TS**al / (plant.mcp * plant.phys['tc']**(al - 1)); cb = (TS / th['tau_s'])**be
    N = len(Q1); wa, wb = gl_w(al, N + 1), gl_w(be, N + 1); X = np.zeros((N + 1, 4))
    for k in range(1, N + 1):
        xp = X[k - 1]; f1, f2 = plant.F(xp[0], xp[1], Q1[k - 1], Q2[k - 1])
        hist = X[k - 1::-1]; n = hist.shape[0]
        ma = wa[1:n + 1] @ hist[:, :2]; mb = wb[1:n + 1] @ hist[:, 2:]
        X[k] = [ca * f1 - ma[0], ca * f2 - ma[1], cb * (xp[0] - xp[2]) - mb[0], cb * (xp[1] - xp[3]) - mb[1]]
    return plant.Ta + X[:N, 2]

_s = scenario(300, 300); _N = _s['N']
_q = np.where(np.arange(_N) >= 20, 40.0, 0.0)
_eq0 = dict(Tbar=PL_D.Ta, u=0.0, q2=0.0, x=np.zeros(4))
_P0 = torch.tensor([[1.0, 0.01, 1.0, 1.0, 1.0, 0.1]])
with torch.no_grad():
    o = closed_loop(PL_D, _P0, [_eq0], _s, u_open=_q[None, :])
    _ref = sim_frac_np(PL_D, _q, np.zeros(_N) + _s['q2_step'])
    e1 = PL_D.equilibrium(SC['ops'][0])
    o2 = closed_loop(PL_D, _P0, [e1], scenario(300, 1), u_open=np.full((1, 301), e1['u']))
print('máx |torch − NB02| = %.2e °C' % np.abs(o['y'][0, 0].numpy() - _ref).max(),
      '| deriva en equilibrio = %.2e °C' % (o2['y'][0, 0] - e1['Tbar']).abs().max().item())

## 4. Robustez en frecuencia: Ms y margen de fase diferenciables

C(jω) = Kp + Ki (jω)^−λ + Kd (jω)^μ /(Tf jω + 1) con (jω)^a = ω^a e^{jaπ/2} (sin Oustaloup). Para el costo se usa la cota
suave Ms ≈ κ⁻¹ log Σ exp(κ|S|) (κ = 200, sobreestima ≤ log(n_ω)/κ ≈ 0,03) y el margen de fase interpolado en el cruce de
ganancia. Para los reportes se usa el máximo exacto en una rejilla densa.

In [ ]:
def C_jw(P, w):
    # P (C, 6) -> C(jω) (C, n_w), complejo
    w = torch.as_tensor(w); jw = lambda a: w[None, :] ** a[:, None] * torch.exp(1j * a[:, None] * math.pi / 2)
    Kp, Ki, Kd, lam, mu = (P[:, i] for i in range(5))
    return (Kp[:, None] + Ki[:, None] * jw(-lam) + Kd[:, None] * jw(mu) / (CTRL['Tf'] * 1j * w[None, :] + 1))

def loop_metrics(P, G, w, smooth=True):
    # G (J, n_w) complejo numpy. Devuelve Ms (C, J), PM [°] (C, J), wc (C, J)
    G = torch.as_tensor(G); Cw = C_jw(P, w)                     # (C, n_w)
    Lw = Cw[:, None, :] * G[None, :, :]                         # (C, J, n_w)
    S = 1.0 / (1.0 + Lw); aS = S.abs()
    Ms = (torch.logsumexp(ROB['kappa'] * aS, -1) / ROB['kappa']) if smooth else aS.max(-1).values
    # fase: ángulo de C (siempre en (−π, π)) + fase desenvuelta de G
    phG = torch.as_tensor(np.unwrap(np.angle(G.numpy()), axis=-1))
    ph = torch.angle(Cw)[:, None, :] + phG[None]
    lm = torch.log(Lw.abs()); lw = torch.log(torch.as_tensor(w))
    PM = torch.full(Ms.shape, 180.0); wc = torch.full(Ms.shape, float('nan'))
    sign = (lm[..., :-1] >= 0) & (lm[..., 1:] < 0)              # cruces descendentes de |L| = 1
    for c in range(Ms.shape[0]):
        for j in range(Ms.shape[1]):
            idx = torch.nonzero(sign[c, j]).flatten()
            if len(idx) == 0: continue
            pms = []
            for n_, i in enumerate(idx):
                t = lm[c, j, i] / (lm[c, j, i] - lm[c, j, i + 1])
                phi = ph[c, j, i] + t * (ph[c, j, i + 1] - ph[c, j, i])
                if n_ == 0:     # primer cruce: PM con signo (fase desenvuelta desde baja frecuencia)
                    pm_ = 180.0 + phi * 180 / math.pi
                else:           # cruces siguientes: distancia angular a −1 (fase llevada a (−π, π])
                    pm_ = 180.0 - torch.abs(torch.remainder(phi + math.pi, 2 * math.pi) - math.pi) * 180 / math.pi
                pms.append((pm_, torch.exp(lw[i] + t * (lw[i + 1] - lw[i]))))
            pm, wcc = min(pms, key=lambda z: float(z[0].detach()))
            PM = PM.index_put((torch.tensor([c]), torch.tensor([j])), pm.reshape(1))
            wc = wc.index_put((torch.tensor([c]), torch.tensor([j])), wcc.reshape(1).detach())
    return Ms, PM, wc

def G_plant(pl, eqs, w): return np.array([pl.G11(w, e) for e in eqs])

EQ_D = [PL_D.equilibrium(T) for T in SC['ops']]
G_D, G_D_dense = G_plant(PL_D, EQ_D, W), G_plant(PL_D, EQ_D, W_DENSE)
EQ_B3 = [PL_B3.equilibrium(T) for T in SC['ops']]
G_B3 = G_plant(PL_B3, EQ_B3, W)
if MS_REF_B3 == 'both': G_B3 = np.r_[G_B3, G_D]      # filas = puntos acotados (propios + sustituto de diseño)

## 5. Parametrización y comparadores analíticos (B4 SIMC, B1 tipo Monje)

p = (Kp, Ki, Kd, λ, μ, Kt) se optimiza como z = (log Kp, log Ki, log Kd, logit(λ/2), logit(μ/2), log Kt) (λ, μ ∈ (0, 2)).
**B4:** SIMC sobre el SOPDT (Kc = τ1/(k(τc+θ)), τI = min(τ1, 4(τc+θ)), τD = τ2), pasado a forma paralela; τc se busca
para que Ms = Ms_max **sobre su propio modelo** (SOPDT con retardo); Kt = 1/Kp (back-calculation con Tt = τI).
**B1:** FOPID sobre el FFOPDT con especificaciones tipo Monje et al. (2008): PM = 60° en ωc, fase plana en ωc (robustez
a la ganancia), |T| ≤ −20 dB para ω ≥ 10ωc, |S| ≤ −20 dB para ω ≤ ωc/10 y Ms ≤ Ms_max **sobre el FFOPDT**
(`MS_REF_B1 = 'own'`), resuelto como mínimos cuadrados penalizados en frecuencia (sin simular el lazo). ωc se busca:
se empieza en el ωc de B4 y se baja hasta que todas las especificaciones se cumplen.

In [ ]:
NAMES = ['Kp', 'Ki', 'Kd', 'lam', 'mu', 'Kt']
def to_nat(z):
    z = torch.atleast_2d(z)
    return torch.stack([torch.exp(z[:, 0]), torch.exp(z[:, 1]), torch.exp(z[:, 2]), 2 * torch.sigmoid(z[:, 3]),
                        2 * torch.sigmoid(z[:, 4]), torch.exp(z[:, 5])], 1)
def to_z(p):
    p = np.asarray(p, float); lg = lambda o: math.log(o / 2 / (1 - o / 2))
    return torch.tensor([math.log(p[0]), math.log(p[1]), math.log(max(p[2], 1e-6)), lg(p[3]), lg(p[4]), math.log(p[5])])
def pdict(P): return dict(zip(NAMES, [float(v) for v in torch.as_tensor(P).flatten()]))

def simc_pid(sp, tau_c):
    K, t1, t2, th = sp['K'], sp['tau1'], sp['tau2'], sp['theta']
    Kc = t1 / (K * (tau_c + th)); tI = min(t1, 4 * (tau_c + th)); tD = t2
    Kp = Kc * (1 + tD / tI); Ki = Kc / tI; Kd = Kc * tD           # serie -> paralelo
    return [Kp, Ki, Kd, 1.0, 1.0, 1.0 / Kp]

def Ms_exact(p, G, w=W_DENSE):
    with torch.no_grad():
        Ms, PM, wc = loop_metrics(torch.tensor([p]), G, w, smooth=False)
    return Ms[0], PM[0], wc[0]

G_SOP = sopdt_G(SOPDT, W_DENSE)[None, :]
G_MSREF_B4 = {'own': [G_SOP], 'design': [G_D_dense], 'both': [G_SOP, G_D_dense]}[MS_REF]
def ms_b4(tc): return max(float(Ms_exact(simc_pid(SOPDT, tc), G)[0].max()) for G in G_MSREF_B4)
try:
    tc_b4 = brentq(lambda tc: ms_b4(tc) - ROB['Ms_max'], 0.5, 5000)
except ValueError:                      # Ms_max inalcanzable en el rango: el τc más agresivo que lo respeta
    tcs = np.logspace(np.log10(0.5), np.log10(5000), 60); ok = [tc for tc in tcs if ms_b4(tc) <= ROB['Ms_max']]
    tc_b4 = ok[0] if ok else tcs[-1]; print('Aviso: Ms(τc) no cruza Ms_max; τc =', round(tc_b4, 1))
P_B4 = simc_pid(SOPDT, tc_b4)
Ms4, PM4, wc4 = Ms_exact(P_B4, G_SOP); Ms4d, PM4d, _ = Ms_exact(P_B4, G_D_dense)
print(f'B4 SIMC (Ms_max sobre {MS_REF}): τc = {tc_b4:.1f} s ->', {k: round(v, 4) for k, v in pdict(P_B4).items()},
      f'| en SOPDT: Ms={float(Ms4[0]):.3f} PM={float(PM4[0]):.1f}° ωc={float(wc4[0]):.4f} rad/s'
      f' | en el sustituto (peor OP): Ms={float(Ms4d.max()):.3f} PM={float(PM4d.min()):.1f}°')

def tune_b1():
    # Monje et al. (2008): ωc y PM especificados, fase plana en ωc, |T| alto y |S| bajo acotados; aquí además Ms <= Ms_max.
    # ωc no se conoce de antemano: se prueban ωc decrecientes (desde el de B4) y se queda el mayor que cumple todo.
    Gw = torch.as_tensor(ffopdt_G(FFOPDT, W)); G_own = ffopdt_G(FFOPDT, W)[None, :]
    G_ms = {'own': G_own, 'design': G_D, 'both': np.r_[G_own, G_D]}[MS_REF_B1]  # filas = modelos/puntos acotados
    mask = torch.tensor([1., 1., 1., 1., 1., 0.]); best = None
    # arranque: PID SIMC sobre el FFOPDT aproximado como SOPDT por la regla de la mitad (τ = τa^(1/α), θ/2 a τ2)
    al_f = FFOPDT.get('alpha_f', FFOPDT.get('alpha')); tau_eq = FFOPDT['tau_a'] ** (1 / al_f)
    sop_eq = dict(K=FFOPDT['K'], tau1=tau_eq, tau2=FFOPDT['theta'] / 2, theta=FFOPDT['theta'] / 2)
    z_start = to_z(simc_pid(sop_eq, FFOPDT['theta']))
    db = lambda x: 20 * torch.log10(x)
    for fac in [1.0, 0.7, 0.5, 0.35, 0.25, 0.18, 0.12]:
        wc_t = float(wc4[0]) * fac
        wcs = np.array([wc_t * 0.95, wc_t, wc_t * 1.05]); Gc = torch.as_tensor(ffopdt_G(FFOPDT, wcs))
        lo, hi = torch.as_tensor(W < wc_t / 10), torch.as_tensor(W > wc_t * 10)
        phGc = torch.as_tensor(np.interp(np.log(wcs), np.log(W), np.unwrap(np.angle(ffopdt_G(FFOPDT, W)))))
        z = z_start.clone().requires_grad_(True)
        def cost(parts=False):
            P = to_nat(z); Lc = C_jw(P, wcs)[0] * Gc
            ph = torch.angle(C_jw(P, wcs)[0]) + phGc       # fase de L desenvuelta desde baja frecuencia
            Ms, PMl, _ = loop_metrics(P, G_ms, W)
            Lw = C_jw(P, W)[0] * Gw; Sw = (1 / (1 + Lw)).abs(); Tw = (Lw / (1 + Lw)).abs()
            r = dict(gain=10 * torch.log(Lc[1].abs()) ** 2, pm=((180 + ph[1] * 180 / math.pi - B1_SPEC['PM']) / 5) ** 2,
                     flat=(((ph[2] - ph[0]) * 180 / math.pi) / 5) ** 2 if B1_SPEC['flat'] else torch.tensor(0.0),
                     T=torch.relu(db(Tw[hi]) - B1_SPEC['T_hf_dB']).pow(2).mean() / 10 if hi.any() else torch.tensor(0.0),
                     S=torch.relu(db(Sw[lo]) - B1_SPEC['S_lf_dB']).pow(2).mean() / 10 if lo.any() else torch.tensor(0.0),
                     Ms=COST['rho'] * (torch.relu(Ms - ROB['Ms_max']) ** 2).sum(),
                     PMmin=COST['rho'] * (torch.relu((ROB['PM_min'] - PMl) / 10) ** 2).sum())   # evita lazos condicionalmente estables
            return (sum(r.values()), r) if parts else sum(r.values())
        opt = torch.optim.Adam([z], lr=0.05)
        for it in range(300 if MODE == 'full' else 80):
            opt.zero_grad(); L = cost(); L.backward(); z.grad *= mask; opt.step()
        zm = z.detach()[mask > 0].clone().requires_grad_(True); z0 = z.detach().clone()
        opt = torch.optim.LBFGS([zm], max_iter=100 if MODE == 'full' else 20, line_search_fn='strong_wolfe')
        def closure():
            nonlocal z
            opt.zero_grad(); z = z0.masked_scatter(mask > 0, zm); L = cost(); L.backward(); return L
        try: opt.step(closure)
        except Exception: pass
        z = z0.masked_scatter(mask > 0, zm.detach()); L, r = cost(True)
        p = to_nat(z.detach())[0].numpy().copy(); p[5] = 1.0 / p[0]
        G_own_d = ffopdt_G(FFOPDT, W_DENSE)[None, :]
        Ms_, PM_, wc_ = Ms_exact(list(p), G_own_d)
        Msr = Ms_exact(list(p), {'own': G_own_d, 'design': G_D_dense, 'both': np.r_[G_own_d, G_D_dense]}[MS_REF_B1])[0]
        ok = float(L) < 0.5 and float(Msr.max()) <= ROB['Ms_max'] + 0.02 and float(PM_[0]) > 0
        print(f'   B1 ωc={wc_t:.4f}: residuo {float(L):.3g} ({", ".join(f"{k} {float(v):.2g}" for k, v in r.items())}) '
              f'PM={float(PM_[0]):.1f}° Ms={float(Msr.max()):.2f} -> {"cumple" if ok else "no cumple"}')
        if best is None or (ok and not best[2]) or (ok == best[2] and float(L) < best[1]): best = (list(p), float(L), ok, wc_t)
        if ok: break
    if not best[2]: print('Aviso: ninguna ωc cumplió todas las especificaciones de B1; se usa la de menor residuo')
    return best[0], best[1], best[3]

t0 = time.time(); P_B1, jb1, wc_b1 = tune_b1()
Ms1, PM1, wc1 = Ms_exact(P_B1, ffopdt_G(FFOPDT, W_DENSE)[None, :])
print(f'B1 Monje/FFOPDT ({time.time() - t0:.0f} s, ωc objetivo {wc_b1:.4f}, residuo {jb1:.3g}):',
      {k: round(v, 4) for k, v in pdict(P_B1).items()},
      f'| en FFOPDT: Ms={float(Ms1[0]):.3f} PM={float(PM1[0]):.1f}° ωc={float(wc1[0]):.4f}')

## 6. Costo J(p) = J_perf + ρ J_rob y sintonía por gradiente

J_perf = (1/J) Σ_j [w_r IAE_r,j/IAE_r,j^B4 + w_d IAE_d,j/IAE_d,j^B4 + w_u TV_j/TV_j^B4] (normalizado con B4 en el modelo
de diseño de cada controlador, así J_perf(B4) = w_r + w_d + w_u). J_rob = Σ_j [Ms_j − Ms_max]₊² + [(PM_min − PM_j)/10°]₊².
Adam (n_adam) → L-BFGS (n_lbfgs) desde el SIMC (λ = μ = 1) y `n_starts − 1` arranques perturbados; se queda el mejor.
Se cuenta el número de evaluaciones del costo y el tiempo, para comparar con la PSO (B2).

In [ ]:
class Problem:
    def __init__(self, plant, G, name):
        self.plant, self.G, self.name = plant, G, name
        self.eqs = [plant.equilibrium(T) for T in SC['ops']]
        with torch.no_grad():
            self.ref = perf_terms(closed_loop(plant, torch.tensor([P_B4]), self.eqs))   # normalización con B4
        self.n_eval = 0

    def terms(self, P, rob=True):
        out = closed_loop(self.plant, P, self.eqs); m = perf_terms(out)
        Jp = (COST['w_r'] * m['IAE_r'] / self.ref['IAE_r'] + COST['w_d'] * m['IAE_d'] / self.ref['IAE_d'] +
              COST['w_u'] * m['TV'] / self.ref['TV']).mean(-1)
        Ms, PM, wc = loop_metrics(P, self.G, W)
        Jr = (torch.relu(Ms - ROB['Ms_max']) ** 2 + torch.relu((ROB['PM_min'] - PM) / 10) ** 2).sum(-1)
        self.n_eval += P.shape[0]
        return Jp, Jr, Ms, PM, m

    def cost(self, P, rho):
        Jp, Jr, *_ = self.terms(P); return Jp + rho * Jr, Jp, Jr

def tune_grad(prob, z0, free=(1, 1, 1, 1, 1, 1), rho=COST['rho'], label='', n_starts=None):
    mask = torch.tensor(free, dtype=torch.float64)
    rng = np.random.default_rng(OPT['seed']); best, hist = None, []
    t0 = time.time(); prob.n_eval = 0; n_grad = 0
    for s in range(n_starts or OPT['n_starts_other']):
        z_init = z0.clone()
        if s > 0: z_init = z_init + mask * torch.tensor(rng.normal(0, OPT['start_sd'], 6))
        zf = z_init[mask > 0].clone().requires_grad_(True)
        full = lambda: z_init.masked_scatter(mask > 0, zf)
        def J():
            nonlocal n_grad
            L, Jp, Jr = prob.cost(to_nat(full()), rho); n_grad += 1
            return L, Jp, Jr
        opt = torch.optim.Adam([zf], lr=OPT['lr'])
        for it in range(OPT['n_adam']):
            opt.zero_grad(); L, Jp, Jr = J(); L.backward(); opt.step()
            hist.append(dict(start=s, stage='adam', it=it, J=float(L.detach()), J_perf=float(Jp.detach()), J_rob=float(Jr.detach()),
                             t_s=time.time() - t0, **pdict(to_nat(full().detach()))))
        if OPT['n_lbfgs'] > 0:
            opt = torch.optim.LBFGS([zf], lr=1.0, max_iter=OPT['n_lbfgs'], history_size=20, line_search_fn='strong_wolfe')
            def closure():
                opt.zero_grad(); L, *_ = J(); L.backward(); return L
            try: opt.step(closure)
            except Exception as ex: print('L-BFGS se detuvo:', ex)
        with torch.no_grad(): L, Jp, Jr = prob.cost(to_nat(full()), rho)
        hist.append(dict(start=s, stage='lbfgs', it=OPT['n_adam'], J=float(L), J_perf=float(Jp), J_rob=float(Jr),
                         t_s=time.time() - t0, **pdict(to_nat(full().detach()))))
        print(f'  {label} arranque {s}: J = {float(L):.4f} (J_perf {float(Jp):.4f}, J_rob {float(Jr):.2e})',
              {k: round(v, 4) for k, v in pdict(to_nat(full().detach())).items()})
        if best is None or float(L) < best[0]: best = (float(L), full().detach().clone())
    dt = time.time() - t0
    return to_nat(best[1])[0].tolist(), pd.DataFrame(hist), dict(time_s=dt, n_eval=prob.n_eval, n_grad=n_grad,
                                                                   ms_per_grad=1000 * dt / max(n_grad, 1))

def tune_pso(prob, z0, rho=COST['rho'], label='B2'):
    rng = np.random.default_rng(PSO['seed']); n = PSO['n_part']
    span = np.array([math.log(5)] * 3 + [2.2, 2.2] + [math.log(5)])
    lo = z0.numpy() - span; hi = z0.numpy() + span
    lo[3:5], hi[3:5] = -2.2, 2.2
    X = rng.uniform(lo, hi, (n, 6)); X[0] = z0.numpy(); V = np.zeros_like(X)
    t0 = time.time(); prob.n_eval = 0
    def f(X):
        with torch.no_grad(): L, _, _ = prob.cost(to_nat(torch.tensor(X)), rho)
        return L.numpy()
    fx = f(X); pb, pbf = X.copy(), fx.copy(); g = pb[pbf.argmin()].copy(); hist = []
    for it in range(PSO['n_iter']):
        r1, r2 = rng.random((n, 6)), rng.random((n, 6))
        V = PSO['w'] * V + PSO['c1'] * r1 * (pb - X) + PSO['c2'] * r2 * (g - X)
        X = np.clip(X + V, lo, hi); fx = f(X)
        imp = fx < pbf; pb[imp], pbf[imp] = X[imp], fx[imp]; g = pb[pbf.argmin()].copy()
        hist.append(dict(it=it, J_best=float(pbf.min()), t_s=time.time() - t0))
    print(f'  {label} PSO: J = {pbf.min():.4f} ({prob.n_eval} evaluaciones, {time.time() - t0:.0f} s)')
    return to_nat(torch.tensor(g))[0].tolist(), pd.DataFrame(hist), dict(time_s=time.time() - t0, n_eval=prob.n_eval,
                                                                        n_grad=0, ms_per_grad=float('nan'))

## 7. Sintonía de P, comparadores y ablaciones (se guarda y retoma)

Orden: B4 → B1 (ya calculados) → **P** → A1 (λ = μ = 1; λ = μ = regla) → A2 (sin J_rob) → B3 (PINN entera) → B2 (PSO).
Cada controlador se agrega a `controladores.json` en cuanto termina; al volver a correr la celda se salta lo ya hecho
(borrar el JSON o poner `OVERWRITE = True` para rehacer, p. ej. si se cambia `DESIGN_MODEL`).

In [ ]:
OVERWRITE = False
CJ = os.path.join(OUT_DIR, 'controladores.json')
CTRLS = json.load(open(CJ)) if (os.path.exists(CJ) and not OVERWRITE) else {}
SIG = [DESIGN_TAG if SOURCE == 'kit' else SIM_TAG, DESIGN_MODEL, MODE, CTRL['Tf']]
if CTRLS and CTRLS.get('_design', [])[:3] + [CTRLS.get('_Tf', 5.0)] != SIG:
    print('controladores.json es de otra configuración; se rehace.'); CTRLS = {}
CTRLS['_design'] = SIG[:3]; CTRLS['_Tf'] = CTRL['Tf']
if 'B3' in CTRLS and CTRLS['B3'].get('ms_ref', 'own') != MS_REF_B3:
    print('B3 se rehace: cambió MS_REF_B3'); del CTRLS['B3']

def save_ctrl(key, p, info, desc):
    CTRLS[key] = dict(p=pdict(p), Tf=CTRL['Tf'], h=TS, desc=desc, **info)
    json.dump(CTRLS, open(CJ, 'w'), indent=1)

PROB_D = Problem(PL_D, G_D, 'diseño')
z_simc = to_z(P_B4)
if 'B4' not in CTRLS: save_ctrl('B4', P_B4, dict(tau_c=tc_b4, time_s=0, n_eval=0), 'PID SIMC sobre SOPDT, τc para Ms=Ms_max')
if 'B1' not in CTRLS: save_ctrl('B1', P_B1, dict(time_s=0, n_eval=0), 'FOPID tipo Monje sobre FFOPDT')

RUNS = [('P',  'FOPID, gradiente sobre el sustituto (propuesto)', PROB_D, (1, 1, 1, 1, 1, 1), COST['rho'], z_simc),
        ('A1_PID', 'A1: λ=μ=1 fijos, gradiente sobre el sustituto', PROB_D, (1, 1, 1, 0, 0, 1), COST['rho'], z_simc),
        ('A1_rule', f'A1: λ=μ=α̂={ALPHA_HAT:.3f} fijos', PROB_D, (1, 1, 1, 0, 0, 1), COST['rho'],
         to_z(P_B4[:3] + [min(ALPHA_HAT, 1.95), min(ALPHA_HAT, 1.95), P_B4[5]])),
        ('A2', 'A2: FOPID sin J_rob (ρ=0)', PROB_D, (1, 1, 1, 1, 1, 1), 0.0, z_simc)]
HIST = {}
for key, desc, prob, free, rho, z0 in RUNS:
    if key in CTRLS: print(key, 'ya está'); continue
    print(f'== {key}: {desc}')
    p, h, info = tune_grad(prob, z0, free, rho, key, OPT['n_starts'] if key == 'P' else None); HIST[key] = h
    h.to_csv(os.path.join(OUT_DIR, f'historial_{key}.csv'), index=False); save_ctrl(key, p, info, desc)
if 'B3' not in CTRLS:
    print('== B3: PID, gradiente sobre', PL_B3.name)
    PROB_B3 = Problem(PL_B3, G_B3, 'B3')
    p, h, info = tune_grad(PROB_B3, z_simc, (1, 1, 1, 0, 0, 1), COST['rho'], 'B3')
    h.to_csv(os.path.join(OUT_DIR, 'historial_B3.csv'), index=False)
    save_ctrl('B3', p, dict(info, ms_ref=MS_REF_B3), f'PID gradiente sobre {PL_B3.name} (Ms/PM: {MS_REF_B3})')
if 'B2' not in CTRLS:
    print('== B2: FOPID, PSO sobre el sustituto de diseño')
    p, h, info = tune_pso(PROB_D, z_simc)
    h.to_csv(os.path.join(OUT_DIR, 'historial_B2.csv'), index=False); save_ctrl('B2', p, info, 'FOPID PSO, mismo costo')
ORDER = ['P', 'B1', 'B2', 'B3', 'B4', 'A1_PID', 'A1_rule', 'A2']
tabc = pd.DataFrame([dict(id=k, **CTRLS[k]['p'], time_s=CTRLS[k].get('time_s'), n_eval=CTRLS[k].get('n_eval'),
                          ms_per_grad=CTRLS[k].get('ms_per_grad')) for k in ORDER])
tabc.to_csv(os.path.join(OUT_DIR, 'controladores.csv'), index=False); print(tabc.round(4).to_string(index=False))

## 8. Evaluación: sustituto de diseño y plantas de verificación

Para cada controlador y planta: IAE_r, IAE_d, TV (suma sobre puntos de operación), Ms y PM exactos (peor caso sobre los
puntos de operación) sobre la linealización de esa planta, y J_perf normalizado con B4 **en esa misma planta**.
ΔJ = (J_planta − J_diseño)/J_diseño mide la transferencia (H3), con J_perf normalizado con B4 en el diseño.

In [ ]:
PLANTS = [('diseño', PL_D)] + [(f'verif{i + 1}', pl) for i, pl in enumerate(PL_V)]
rows = []
Pall = torch.tensor([[CTRLS[k]['p'][n] for n in NAMES] for k in ORDER])
for pname, pl in PLANTS:
    eqs = [pl.equilibrium(T) for T in SC['ops']]
    with torch.no_grad():
        m = perf_terms(closed_loop(pl, Pall, eqs))
        Ms, PM, wc = loop_metrics(Pall, G_plant(pl, eqs, W_DENSE), W_DENSE, smooth=False)
    for c, k in enumerate(ORDER):
        for j, T in enumerate(SC['ops']):
            rows.append(dict(planta=pname, modelo=pl.name, id=k, op=T, IAE_r=float(m['IAE_r'][c, j]),
                             IAE_d=float(m['IAE_d'][c, j]), TV=float(m['TV'][c, j]), Ms=float(Ms[c, j]),
                             PM=float(PM[c, j]), wc=float(wc[c, j])))
met = pd.DataFrame(rows); met.to_csv(os.path.join(OUT_DIR, 'metricas_por_op.csv'), index=False)

ref = met[met.id == 'B4'].set_index(['planta', 'op'])
def jn(r):
    b = ref.loc[(r.planta, r.op)]
    return COST['w_r'] * r.IAE_r / b.IAE_r + COST['w_d'] * r.IAE_d / b.IAE_d + COST['w_u'] * r.TV / b.TV
met['J_perf_rel_B4'] = met.apply(jn, axis=1)
res = met.groupby(['planta', 'modelo', 'id'], sort=False).agg(IAE_r=('IAE_r', 'sum'), IAE_d=('IAE_d', 'sum'), TV=('TV', 'sum'),
                                                              Ms=('Ms', 'max'), PM=('PM', 'min'),
                                                              J_perf_rel_B4=('J_perf_rel_B4', 'mean')).reset_index()
# ΔJ con la normalización del diseño (B4 en el sustituto de diseño)
refD = met[(met.id == 'B4') & (met.planta == 'diseño')].set_index('op')
met['J_perf_normD'] = met.apply(lambda r: COST['w_r'] * r.IAE_r / refD.loc[r.op].IAE_r + COST['w_d'] * r.IAE_d / refD.loc[r.op].IAE_d +
                                COST['w_u'] * r.TV / refD.loc[r.op].TV, axis=1)
jd = met.groupby(['planta', 'id']).J_perf_normD.mean()
res['dJ_transfer_pct'] = [100 * (jd[(r.planta, r.id)] - jd[('diseño', r.id)]) / jd[('diseño', r.id)] for r in res.itertuples()]
res['lam'] = res.id.map(lambda k: CTRLS[k]['p']['lam']); res['mu'] = res.id.map(lambda k: CTRLS[k]['p']['mu'])
res.to_csv(os.path.join(OUT_DIR, 'resumen.csv'), index=False)
pd.set_option('display.width', 200); print(res.round(3).to_string(index=False))

# Filas LaTeX: tabla de ablaciones (tab:sim_abl) y de comparación en el sustituto de diseño
tex = []
for k, lab in [('A1_PID', 'A1: $\\lambda=\\mu=1$'), ('A1_rule', 'A1: rule'), ('P', 'A1: learnable (proposed)'),
               ('A2', 'A2: without $J_{\\text{rob}}$')]:
    r = res[(res.planta == ('verif1' if SOURCE == 'sim' else 'diseño')) & (res.id == k)].iloc[0]
    tex.append(f'{lab} & {r.lam:.2f} & {r.mu:.2f} & {r.IAE_r:.0f} & {r.IAE_d:.0f} & {r.TV:.0f} & {r.Ms:.2f} \\\\')
open(os.path.join(OUT_DIR, 'tab_ablaciones_rows.tex'), 'w').write('\n'.join(tex) + '\n')

## 9. Regla de decisión (P frente a B4)

"P supera claramente a B4" si, en el sustituto de diseño **y** en todas las plantas de verificación, IAE_d(P) es al menos
`DECISION_GAIN` % menor con Ms(P) ≤ max(Ms(B4), Ms_max) + 0,05 (es decir, sin pagar robustez por encima de la cota común),
**o** Ms(P) es al menos `DECISION_MS` menor con IAE_d(P) ≤ IAE_d(B4).
Si no → evaluar el caso 2×2 MIMO en simulación (memoria del proyecto).

In [ ]:
dec = []
for pname in res.planta.unique():
    p = res[(res.planta == pname) & (res.id == 'P')].iloc[0]; b = res[(res.planta == pname) & (res.id == 'B4')].iloc[0]
    gain = 100 * (b.IAE_d - p.IAE_d) / b.IAE_d
    ok = (gain >= DECISION_GAIN and p.Ms <= max(b.Ms, ROB['Ms_max']) + 0.05) or (b.Ms - p.Ms >= DECISION_MS and p.IAE_d <= b.IAE_d)
    dec.append(dict(planta=pname, modelo=p.modelo, IAE_d_P=p.IAE_d, IAE_d_B4=b.IAE_d, mejora_IAE_d_pct=gain,
                    Ms_P=p.Ms, Ms_B4=b.Ms, IAE_r_P=p.IAE_r, IAE_r_B4=b.IAE_r, P_claramente_mejor=bool(ok)))
dec = pd.DataFrame(dec); dec.to_csv(os.path.join(OUT_DIR, 'decision_P_vs_B4.csv'), index=False)
print(dec.round(3).to_string(index=False))
VERDICT = ('P supera claramente a B4 -> seguir con el artículo SISO (NB04 en el kit)' if dec.P_claramente_mejor.all() else
           'P NO supera claramente a B4 -> evaluar 2x2 MIMO en simulación antes de decidir (regla del proyecto)')
if MODE == 'quick': VERDICT += '  [modo quick: NO usar para decidir]'
print('\nDECISIÓN:', VERDICT)
open(os.path.join(OUT_DIR, 'decision.txt'), 'w').write(VERDICT + '\n')

## 10. Figuras

In [ ]:
cols = dict(P='C3', B1='C1', B2='C4', B3='C2', B4='k', A1_PID='C0', A1_rule='C9', A2='C5')
j = len(SC['ops']) // 2; T = SC['ops'][j]
with torch.no_grad(): o = closed_loop(PL_D, Pall, EQ_D)
t = np.arange(SCN['N']) * TS
fig, ax = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
ax[0].plot(t, T + SCN['r_step'], c='0.6', lw=1, ls='--', label='r')
for c, k in enumerate(ORDER):
    if k.startswith('A'): continue
    ax[0].plot(t, o['y'][c, j], c=cols[k], lw=1.2 if k == 'P' else 0.9, label=k); ax[1].plot(t, o['u'][c, j], c=cols[k], lw=0.9)
ax[0].axvline(SCN['N_r'], c='0.8', lw=0.8); ax[1].axvline(SCN['N_r'], c='0.8', lw=0.8)
ax[0].set_ylabel('T1 [°C]'); ax[1].set_ylabel('Q1 [%]'); ax[1].set_xlabel('t [s]'); ax[0].legend(ncol=6, fontsize=8)
ax[0].set_title(f'Sustituto de diseño ({PL_D.name}), T̄ = {T:.0f} °C: referencia +{SC["dr"]:.0f} °C y Q2 +{SC["dq2"]:.0f} % en t = {SCN["N_r"]} s')
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'fig01_respuestas.png'), dpi=150)
plt.savefig(os.path.join(OUT_DIR, 'fig01_respuestas.pdf')); plt.show(); plt.close('all')

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
with torch.no_grad():
    for c, k in enumerate(ORDER):
        if k not in ('P', 'B1', 'B3', 'B4', 'A2'): continue
        Lw = C_jw(Pall[c:c + 1], W_DENSE)[0] * torch.as_tensor(G_D_dense[j])
        ax[0].semilogx(W_DENSE, 20 * np.log10(np.abs(1 / (1 + Lw.numpy()))), c=cols[k], label=k)
        ax[1].plot(Lw.real, Lw.imag, c=cols[k], label=k)
ax[0].axhline(20 * np.log10(ROB['Ms_max']), c='0.5', ls='--', lw=0.8); ax[0].set_xlabel('ω [rad/s]'); ax[0].set_ylabel('|S| [dB]')
th_ = np.linspace(0, 2 * np.pi, 200); R = 1 / ROB['Ms_max']
ax[1].plot(-1 + R * np.cos(th_), R * np.sin(th_), c='0.5', ls='--', lw=0.8); ax[1].plot(-1, 0, 'k+')
ax[1].set_xlim(-3, 1); ax[1].set_ylim(-3, 1); ax[1].set_aspect('equal'); ax[1].legend(fontsize=8); ax[1].set_title('Nyquist')
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'fig02_sensibilidad_nyquist.png'), dpi=150); plt.show(); plt.close('all')

hp = os.path.join(OUT_DIR, 'historial_P.csv')
if os.path.exists(hp):
    h = pd.read_csv(hp); fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
    for s, g in h.groupby('start'): ax[0].semilogy(np.arange(len(g)), g.J, label=f'arranque {s}')
    ax[0].set_xlabel('iteración (Adam)'); ax[0].set_ylabel('J'); ax[0].legend(fontsize=7)
    g = h[h.start == h.start.min()]; ax[1].plot(g.lam.values, label='λ'); ax[1].plot(g.mu.values, label='μ')
    ax[1].axhline(1, c='0.6', lw=0.8); ax[1].legend(); ax[1].set_title('órdenes durante la sintonía (P)')
    plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'fig03_convergencia_P.png'), dpi=150); plt.show(); plt.close('all')

## 11. Exportar para NB04 y empaquetar

`controladores_NB04.json`: por controlador, (Kp, Ki, Kd, λ, μ, Kt, Tf, h) y la ley discreta que hay que implementar
en el kit (la misma de la sección 3). Se crea `NB03_<SOURCE>_resultados_<fecha>.zip` con todo `NB03_out/<SOURCE>/`.

In [ ]:
law = ('e_k=r_k-y_k; et_k=e_k+Kt*(u_{k-1}-v_{k-1}); I_k=h^lam*sum_{i=0..k} w_i^(-lam) et_{k-i}; '
       'D_k=filtro_Tf(h^-mu*sum_{i=0..k} w_i^(mu)(y_{k-i}-y_0)), d_k=a d_{k-1}+(1-a) D_k, a=Tf/(Tf+h); '
       'v_k=u_0+Kp e_k+Ki I_k-Kd d_k; u_k=sat_[0,100](v_k). GL w_0=1, w_i=(1-(1+a)/i) w_{i-1}; memoria completa; '
       'u_0 e y_0 = Q1 y T1 en equilibrio al cerrar el lazo.')
json.dump(dict(design=CTRLS['_design'], law=law, Ms_max=ROB['Ms_max'], PM_min=ROB['PM_min'],
               controllers={k: CTRLS[k] for k in ORDER}), open(os.path.join(OUT_DIR, 'controladores_NB04.json'), 'w'), indent=1)
meta = dict(fecha=datetime.datetime.now().isoformat(timespec='seconds'), mode=MODE, source=SOURCE, nb02=str(SUR_SRC),
            design=[DESIGN_TAG, DESIGN_MODEL], b3=B3_MODEL, verif=[pl.name for pl in PL_V], sc={k: v for k, v in SC.items()},
            ctrl=CTRL, rob=ROB, cost=COST, opt=OPT, pso=PSO, b1=B1_SPEC, decision=VERDICT, so=platform.platform(),
            python=platform.python_version(), torch=torch.__version__, procesador=platform.processor(),
            hilos_torch=torch.get_num_threads())
json.dump(meta, open(os.path.join(OUT_DIR, 'metadatos_nb03.json'), 'w'), indent=1, default=str)
zname = f"NB03_{SOURCE}_resultados_{datetime.date.today().strftime('%Y%m%d')}{'_quick' if MODE == 'quick' else ''}.zip"
with zipfile.ZipFile(zname, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT_DIR):
        for fn in files: z.write(os.path.join(root, fn))
print('Guardado:', os.path.abspath(zname), f'({os.path.getsize(zname) / 1e6:.2f} MB)')

## 12. Notas para la discusión

- **Modelo de diseño.** NB02 completo: la fPINN aprende α ≈ 1,04–1,07 y β ≈ 1,00–1,07 (sin fraccionalidad). Entrenada
  con A+B (`AB-C`) valida igual que el mejor físico; entrenada con B+C valida peor que la PINN entera. Decisión (9 nov):
  diseñar sobre la **fPINN reentrenada con A+B+C** (`DESIGN_TAG = 'auto'`), con Ms y PM verificados en el físico entero
  OE-LS. `DESIGN_MODEL`/`DESIGN_TAG` siguen siendo parámetros (cada configuración rehace `controladores.json`;
  renombrar la carpeta `NB03_out/kit` entre corridas si se quieren conservar ambas).
- **Con un sustituto casi entero, cualquier ventaja de P sale del controlador** (λ, μ, sintonía conjunta con robustez),
  no del modelo. Es la lectura de H2; B3 (PID sobre PINN entera) y A1 (λ = μ = 1) son los comparadores clave.
- **Ms a igualdad.** Todos quedan bajo el mismo Ms* = Ms_max: P, B2, B3 y A1–A2 por la penalización sobre su sustituto;
  B4 (τc) y B1 con `MS_REF = 'both'` sobre su propio modelo **y** sobre el sustituto. Ninguno está obligado a llegar
  justo a Ms_max (la cota puede no estar activa), así que la regla de decisión compara contra max(Ms(B4), Ms_max).
  En la prueba, acotar B4 solo sobre el sustituto (`'design'`) dio τc = 5 s y Ms = 3,1 sobre su propio SOPDT (un SIMC
  que nadie usaría); solo sobre el SOPDT (`'own'`) dio Ms = 1,28 sobre el sustituto.
- **Tiempos medidos** (CPU de 4 núcleos de la nube, modo full: 3 puntos × 1800 s): un gradiente ≈ 1,4 s (0,5 s ida +
  0,9 s vuelta); una iteración de la PSO con 40 partículas ≈ 3 s. P (3 arranques) ≈ 25–30 min; A1×2, A2, B3 ≈ 8–10 min
  cada uno; B2 ≈ 5 min; total ≈ 1–1,5 h.
- **No incluido aquí** (pendiente si hace falta): cuantización del PWM (8 bits) y del sensor en el lazo (no
  diferenciables), y la planta virtual con paso 0,25 s en `'sim'` (aquí se simula con h = 1 s).